# GD-CLASS Explorer v1 -- Rigid Inclusion Model

**Date:** February 19, 2026 | **Status:** SUPERSEDED

---

> **HISTORICAL NOTE:** This is the first version of the GD-CLASS Explorer.
> It used the **rigid inclusion** model (kappa = 1/(1-phi) = 1.176) which
> **weakens** early-universe gravity. The reported H_0 = 72.6 was later found
> to be **wrong** due to an inverted formula in compare_planck.py.
> The actual value is H_0 = 65.75 -- GD with rigid inclusions makes the
> Hubble tension **worse**, not better.
>
> **What we learned:** The sign of the kappa modification matters.
> Weakening gravity (kappa > 1) increases r_s, which lowers H_0.
> Strengthening gravity (kappa < 1) is needed to raise H_0.
>
> **See also:**
> - [v2: Compliant Model](GD_CLASS_Explorer_v2_compliant.ipynb) -- kappa = 0.85
> - [v3: Parameter-Fitted](GD_CLASS_Explorer.ipynb) -- kappa = 0.96-0.98, current

---

### Original Description

# GD-CLASS Explorer

**Glassy Dynamics of Spacetime — Interactive Parameter Explorer**

[GitHub repository](https://github.com/lawdroid/class_public/tree/feature/kappa-evolution)

---

### How to use

1. Click **Runtime → Run all** in the menu bar
2. Wait ~30 seconds for all cells to finish
3. Scroll down to the **Interactive Explorer**
4. **Move the sliders** to change GD parameters
5. Plots and results table update automatically

**No Python knowledge required.**

---

### What this notebook computes

GD theory modifies the Friedmann equation with spacetime stiffness κ(z):

| | Equation |
|---|---|
| Standard | H² = (8πG/3) × ρ |
| GD Theory | H² = (8πG/3κ) × ρ_matter_rad + Λ |

The four sliders control:
- **κ(z) profile** — how stiffness evolves with redshift
- **H(z)** — expansion rate compared to ΛCDM
- **r_s** — the sound horizon (acoustic ruler)
- **H₀** — actual Hubble constant today


## Setup
Run this cell to load all required libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64, io, warnings
warnings.filterwarnings('ignore')

# Use a clean plot style
plt.rcParams.update({
    'figure.figsize': (12, 4),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})
print("Setup complete.")

## Load Pre-computed Spectra
Embedded CLASS output and Planck 2018 data (no files needed).

In [ ]:
# Pre-computed CMB power spectra (embedded as compressed data)
_b64 = "..."
_buf = io.BytesIO(base64.b64decode(_b64))
_data = np.load(_buf)

# LCDM baseline (Planck 2018 best-fit, h=0.6736)
l_lcdm = _data['l_lcdm']
Dl_lcdm = _data['Dl_lcdm']

# GD with kappa_c=1.176, beta=0.5, z_onset=10^6, z_freeze=1100
l_gd = _data['l_gd']
Dl_gd = _data['Dl_gd']

# Planck 2018 TT observed data
l_planck = _data['l_planck']
Dl_planck = _data['Dl_planck']
err_lo = _data['err_lo']
err_hi = _data['err_hi']

print(f"Loaded: LCDM ({len(l_lcdm)} pts), GD reference ({len(l_gd)} pts), Planck ({len(l_planck)} pts)")

## Physics Engine
The modified Friedmann equation and sound horizon calculation, implemented in Python.
These reproduce the CLASS background computation for any GD parameters.

In [ ]:
# ===== Cosmological parameters (Planck 2018 best-fit) =====
h = 0.6736
H0 = h * 100.0          # km/s/Mpc
c_light = 299792.458     # km/s
omega_b = 0.02237        # Omega_b h^2
omega_cdm = 0.1200       # Omega_cdm h^2

Omega_b = omega_b / h**2
Omega_cdm = omega_cdm / h**2
Omega_m = Omega_b + Omega_cdm

# Radiation: photons + 3 neutrino species (N_eff = 3.044)
Omega_gamma = 2.469e-5 / h**2       # photon density
Omega_r = Omega_gamma * 1.6914      # total radiation (photons + neutrinos)
Omega_Lambda = 1.0 - Omega_m - Omega_r  # flat universe

# ===== kappa(z) stretched exponential =====
def kappa_of_z(z, kappa_c, z_freeze, z_onset, beta):
    """GD spacetime stiffness as a function of redshift."""
    z = np.atleast_1d(np.float64(z))
    kappa = np.ones_like(z)
    # Below z_freeze: frozen at kappa_c
    mask_low = z <= z_freeze
    kappa[mask_low] = kappa_c
    # Between z_freeze and z_onset: stretched exponential
    mask_mid = (z > z_freeze) & (z < z_onset)
    t = (z[mask_mid] - z_freeze) / (z_onset - z_freeze)
    decay = np.exp(-(t / 0.5)**beta)
    kappa[mask_mid] = 1.0 + (kappa_c - 1.0) * decay
    # Above z_onset: standard gravity (kappa = 1)
    return kappa

# ===== Hubble parameter H(z) =====
def H_of_z(z, kappa_c=1.0, z_freeze=1100, z_onset=1e6, beta=0.5):
    """Hubble parameter in km/s/Mpc with GD modification."""
    z = np.atleast_1d(np.float64(z))
    kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)
    rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3  # matter + radiation
    rho_L = Omega_Lambda                                # Lambda (constant)
    return H0 * np.sqrt(rho_mr / kap + rho_L)

# ===== Sound horizon r_s =====
def compute_rs(kappa_c, z_freeze, z_onset, beta, z_star=1089.8):
    """Comoving sound horizon at recombination [Mpc]."""
    def integrand(z):
        kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
        rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3
        H = H0 * np.sqrt(rho_mr / kap + Omega_Lambda)
        # Baryon loading R = 3*rho_b/(4*rho_gamma)
        R = 3.0 * Omega_b * (1+z)**3 / (4.0 * Omega_gamma * (1+z)**4)
        cs = 1.0 / np.sqrt(3.0 * (1.0 + R))  # sound speed / c
        return cs * c_light / H
    result, _ = quad(integrand, z_star, 1e7, limit=200)
    return result

# ===== Angular diameter distance =====
def compute_DA(kappa_c, z_freeze, z_onset, beta, z_star=1089.8):
    """Comoving angular diameter distance to recombination [Mpc]."""
    def integrand(z):
        return c_light / H_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
    result, _ = quad(integrand, 0, z_star, limit=200)
    return result

# ===== Actual H_0 at z=0 =====
def H0_actual(kappa_c):
    """Actual Hubble constant at z=0 with GD (kappa = kappa_c at z=0)."""
    rho_mr = Omega_r + Omega_m  # at z=0
    return H0 * np.sqrt(rho_mr / kappa_c + Omega_Lambda)

# ===== Age of universe =====
def compute_age(kappa_c, z_freeze, z_onset, beta):
    """Age of universe [Gyr]."""
    def integrand(z):
        kap = kappa_of_z(z, kappa_c, z_freeze, z_onset, beta)[0]
        rho_mr = Omega_r * (1+z)**4 + Omega_m * (1+z)**3
        H = H0 * np.sqrt(rho_mr / kap + Omega_Lambda)
        return 1.0 / ((1+z) * H)
    # Split integral for better convergence
    result = 0.0
    breaks = [0, 1, 10, 100, 1000, 1e4, 1e5, 1e6, 1e7]
    for i in range(len(breaks)-1):
        val, _ = quad(integrand, breaks[i], breaks[i+1], limit=100)
        result += val
    # Convert: result is Mpc*s/km. 1 Mpc = 3.0857e19 km. 1 Gyr = 3.1557e16 s.
    return result * 3.0857e19 / 3.1557e16

# Test: LCDM values
rs_lcdm = compute_rs(1.0, 1100, 1e6, 0.5)
H0_lcdm = H0_actual(1.0)
age_lcdm = compute_age(1.0, 1100, 1e6, 0.5)
print(f"LCDM check:  r_s = {rs_lcdm:.1f} Mpc,  H_0 = {H0_lcdm:.2f} km/s/Mpc,  Age = {age_lcdm:.2f} Gyr")
print(f"(Expected:   r_s ~ 144.5 Mpc,  H_0 = 67.36 km/s/Mpc,  Age ~ 13.8 Gyr)")

---
## Interactive Explorer

**Move the sliders below** to change GD parameters and see how the physics responds.

| Parameter | Physical meaning |
|-----------|-----------------|
| **kappa_c** | Spacetime stiffness at freeze-out. 1.0 = standard gravity (LCDM). 1.176 = Scher-Zallen percolation threshold. |
| **beta** | Stretched exponent controlling transition shape. Small beta = gradual, large beta = sharp. |
| **z_onset** | Redshift where kappa starts departing from 1. Must clear BBN (z ~ 10^9). |
| **z_freeze** | Redshift where kappa freezes at kappa_c. 1100 = recombination. |


In [ ]:
# ===== Create interactive widgets =====
style = {'description_width': '100px'}

slider_kc = widgets.FloatSlider(
    value=1.176, min=1.0, max=1.5, step=0.002,
    description='kappa_c:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.3f'
)
slider_beta = widgets.FloatSlider(
    value=0.5, min=0.1, max=1.0, step=0.05,
    description='beta:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.2f'
)
slider_zonset = widgets.FloatLogSlider(
    value=1e6, min=3, max=8, step=0.1,
    description='z_onset:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.0e'
)
slider_zfreeze = widgets.FloatSlider(
    value=1100, min=100, max=5000, step=50,
    description='z_freeze:', style=style,
    layout=widgets.Layout(width='500px'),
    readout_format='.0f'
)

output = widgets.Output()

def update_plots(change=None):
    kc = slider_kc.value
    beta = slider_beta.value
    zo = slider_zonset.value
    zf = slider_zfreeze.value

    with output:
        clear_output(wait=True)

        # Compute background quantities
        rs = compute_rs(kc, zf, zo, beta)
        rs_ref = compute_rs(1.0, 1100, 1e6, 0.5)  # LCDM reference
        h0 = H0_actual(kc)
        age = compute_age(kc, zf, zo, beta)
        DA = compute_DA(kc, zf, zo, beta)
        DA_ref = compute_DA(1.0, 1100, 1e6, 0.5)
        theta = rs / DA   # angular scale (radians × 1000)
        theta_ref = rs_ref / DA_ref

        # ===== FIGURE =====
        fig = plt.figure(figsize=(14, 12))

        # --- Panel 1: kappa(z) profile ---
        ax1 = fig.add_subplot(2, 2, 1)
        z_arr = np.logspace(0, 7, 500)
        kap = kappa_of_z(z_arr, kc, zf, zo, beta)
        ax1.semilogx(z_arr, kap, 'b-', linewidth=2, label=f'GD (kappa_c={kc:.3f})')
        ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='LCDM (kappa=1)')
        ax1.axvline(x=zf, color='red', linestyle=':', alpha=0.5, label=f'z_freeze={zf:.0f}')
        ax1.axvline(x=zo, color='green', linestyle=':', alpha=0.5, label=f'z_onset={zo:.0e}')
        ax1.axvline(x=1100, color='orange', linestyle='--', alpha=0.3, label='Recombination')
        ax1.set_xlabel('Redshift z')
        ax1.set_ylabel('kappa(z)')
        ax1.set_title('Spacetime Stiffness Profile')
        ax1.legend(fontsize=8, loc='upper left')
        ax1.set_ylim(0.95, max(kc + 0.05, 1.1))

        # --- Panel 2: H(z)/H_LCDM(z) ratio ---
        ax2 = fig.add_subplot(2, 2, 2)
        z_arr2 = np.logspace(0, 5, 300)
        H_gd = H_of_z(z_arr2, kc, zf, zo, beta)
        H_lcdm_arr = H_of_z(z_arr2, 1.0, 1100, 1e6, 0.5)
        ratio = H_gd / H_lcdm_arr
        ax2.semilogx(z_arr2, ratio, 'r-', linewidth=2)
        ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
        ax2.axvline(x=1100, color='orange', linestyle='--', alpha=0.3, label='Recombination')
        ax2.set_xlabel('Redshift z')
        ax2.set_ylabel('H_GD(z) / H_LCDM(z)')
        ax2.set_title('Expansion Rate Ratio')
        ax2.legend(fontsize=8)

        # --- Panel 3: CMB Power Spectrum ---
        ax3 = fig.add_subplot(2, 1, 2)
        ax3.errorbar(l_planck, Dl_planck, yerr=[err_lo, err_hi],
                     fmt='.', color='gray', alpha=0.3, markersize=2,
                     label='Planck 2018', zorder=1)
        ax3.plot(l_lcdm, Dl_lcdm, 'b-', linewidth=1.5, alpha=0.8,
                 label='LCDM (standard)', zorder=2)
        ax3.plot(l_gd, Dl_gd, 'r-', linewidth=1.5, alpha=0.8,
                 label='GD (kappa_c=1.176, beta=0.5)', zorder=3)
        ax3.set_xlabel('Multipole l')
        ax3.set_ylabel('D_l [muK^2]')
        ax3.set_title('CMB TT Power Spectrum (pre-computed CLASS output)')
        ax3.set_xlim(2, 2500)
        ax3.set_ylim(-200, 7000)
        ax3.legend(fontsize=9)

        plt.tight_layout()
        plt.show()

        # ===== RESULTS TABLE =====
        print("=" * 70)
        print("  RESULTS TABLE")
        print("=" * 70)
        print(f"  {'Quantity':<30s} {'LCDM':<15s} {'GD':<15s} {'Change':<10s}")
        print("-" * 70)
        print(f"  {'Sound horizon r_s [Mpc]':<30s} {rs_ref:<15.2f} {rs:<15.2f} {(rs/rs_ref-1)*100:+.1f}%")
        print(f"  {'Ang. diam. dist. D_A [Mpc]':<30s} {DA_ref:<15.1f} {DA:<15.1f} {(DA/DA_ref-1)*100:+.1f}%")
        print(f"  {'theta_* = r_s/D_A':<30s} {theta_ref:<15.4f} {theta:<15.4f} {(theta/theta_ref-1)*100:+.1f}%")
        print(f"  {'H_0 at z=0 [km/s/Mpc]':<30s} {H0_actual(1.0):<15.2f} {h0:<15.2f} {(h0/H0_actual(1.0)-1)*100:+.1f}%")
        print(f"  {'G_eff/G_N (at z < z_freeze)':<30s} {'1.000':<15s} {1.0/kc:<15.5f} {(1.0/kc-1)*100:+.1f}%")
        print(f"  {'Age [Gyr]':<30s} {age_lcdm:<15.2f} {age:<15.2f} {(age/age_lcdm-1)*100:+.1f}%")
        print("=" * 70)
        print()
        if kc > 1.001:
            print("  NOTE: The CMB spectrum plot shows a FIXED pre-computed GD run")
            print("  (kappa_c=1.176, beta=0.5). The background quantities above")
            print("  update with sliders. Full spectrum requires running CLASS.")
        if kc <= 1.001:
            print("  kappa_c ~ 1.0: This is standard LCDM (no GD modification).")

# Connect sliders to update function
for s in [slider_kc, slider_beta, slider_zonset, slider_zfreeze]:
    s.observe(update_plots, names='value')

# Display
print("Move the sliders below, then scroll down to see the plots and results.")
print()
display(widgets.VBox([slider_kc, slider_beta, slider_zonset, slider_zfreeze]))
display(output)

# Initial plot
update_plots()

---
## Understanding the Parameters

### kappa_c (Stiffness at freeze-out)
- **kappa_c = 1.0**: Standard gravity. No GD modification. This IS Lambda-CDM.
- **kappa_c = 1.176**: The Scher-Zallen percolation threshold (phi_c = 0.15, kappa = 1/(1-phi)).
  This is the value predicted by GD theory with zero free parameters.
- **kappa_c > 1**: Weaker effective gravity (G_eff = G_N / kappa < G_N).
  Sound waves travel farther before recombination, increasing the sound horizon r_s.

### beta (Stretched exponent)
- Controls the **shape** of the kappa(z) transition, NOT its magnitude.
- **Small beta (0.3)**: Most of the change happens early (near z_onset), with a long gradual tail.
- **Large beta (0.9)**: Change concentrated in the middle of the transition.
- **In practice, beta barely affects observables** — all values from 0.3 to 0.9 give nearly identical r_s and H_0.

### z_onset (Transition start)
- The redshift where kappa begins departing from 1 (standard gravity).
- **z_onset = 10^6**: Clears Big Bang Nucleosynthesis (BBN at z ~ 10^9) by three orders of magnitude. Nucleosynthesis is completely standard.
- **z_onset = 10^4**: Concentrates the transition near recombination, leaving features in the CMB damping tail.

### z_freeze (Transition end)
- The redshift where kappa freezes at kappa_c.
- **z_freeze = 1100**: Freezes at recombination. The acoustic peak structure is preserved.
- **z_freeze < 1100**: kappa still evolving after recombination, modifying ISW effect and structure growth.

---

## Key Findings from CLASS Simulations

| Finding | Details |
|---------|---------|
| **H_0 with rho/kappa** | H_0 = 65.75 km/s/Mpc (kappa_c=1.176). Tension gets WORSE, not better. |
| **ISW excess** | D_l(l=2) = 10,441 muK^2 vs Planck 225 muK^2 — 46x above observed. |
| **First peak** | 29% below Planck at l=220. |
| **Beta insensitivity** | All beta values (0.3-0.9) give same results within 2%. |
| **Phase C needed** | Background-only modification is insufficient. Consistent perturbations (G_eff in Poisson equation) are essential. |

---

